In [1]:
from transformers import pipeline, set_seed 
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt 
from datasets import load_dataset
import evaluate
metric = evaluate.load("rouge")
import pandas as pd 
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import nltk 
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")



c:\Users\AKSHAY\Music\anaconda\envs\textS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\AKSHAY\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device="cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [6]:
model_ckpt = "google/pegasus-cnn_dailymail"

# 3. Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

print("Tokenizer loaded successfully")

# 4. Load Pegasus model
print("Loading Pegasus model...")
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt)

print("Model loaded successfully")

Loading tokenizer...
Tokenizer loaded successfully
Loading Pegasus model...


Loading weights: 100%|██████████| 680/680 [00:00<00:00, 9134.05it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully


In [7]:
model_pegasus = model_pegasus.to(device)

In [8]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("knkarthick/samsum")

Generating test split: 100%|██████████| 819/819 [00:00<00:00, 16849.88 examples/s]


In [9]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [16]:
ds["train"].column_names
ds["train"][1]['summary']

'Olivia and Olivier are voting for liberals in this election. '

In [23]:
def convert_examples_to_features(example_batch):
    input_encodings =tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)

    target_encodings = tokenizer(
        text_target=example_batch['summary'],max_length=128,truncation=True)

    return {
        'input_ids':input_encodings['input_ids'],
        'attention_mask':input_encodings['attention_mask'],
        'labels':target_encodings['input_ids']
    }    

In [24]:
ds_pt=ds.map(convert_examples_to_features,batched=True)

Map: 100%|██████████| 819/819 [00:00<00:00, 2679.00 examples/s]


In [25]:
ds_pt['train']

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14731
})

In [26]:
from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator=DataCollatorForSeq2Seq(tokenizer,model=model_pegasus)


In [31]:
from transformers import TrainingArguments,Trainer

trainer_args=TrainingArguments(
    output_dir='pegasus-samsum',num_train_epochs=1,warmup_steps=500,
    per_device_train_batch_size=1,per_device_eval_batch_size=1,
    weight_decay=0.01,logging_steps=10,
    eval_strategy='steps',eval_steps=500,save_steps=1e6,
    gradient_accumulation_steps=16
)

In [35]:
trainer=Trainer(model=model_pegasus,args=trainer_args,
                processing_class=tokenizer, data_collator=seq2seq_data_collator,
                train_dataset=ds_pt["test"],
                eval_dataset=ds_pt["validation"])

In [36]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
c:\Users\AKSHAY\Music\anaconda\envs\textS\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


: 

: 